In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("SilverToGold")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "password")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)

In [ ]:
# Ler todos os arquivos da camada Silver no MinIO
df = spark.read.json("s3a://silver/")
df.printSchema()
df.show(truncate=False)

In [3]:
from pyspark.sql.functions import *

df = df.withColumn("timestamp_hour", date_trunc("hour", col("timestamp"))) \
    .withColumn("day", dayofmonth("timestamp")) \
    .withColumn("hour", hour("timestamp"))

In [4]:
gold = df.groupBy(
    "equipment_id",
    "factory_id",
    "measurement_type",
    "timestamp_hour",
    "year",
    "month",
    "day",
    "hour"
).agg(
    avg("value").alias("avg_value"),
    min("value").alias("min_value"),
    max("value").alias("max_value"),
    stddev("value").alias("std_value"),
    count("*").alias("event_count"),
    sum(col("is_anomaly").cast("int")).alias("anomaly_count")
)

In [5]:
gold = gold.withColumn(
    "anomaly_rate",
    col("anomaly_count") / col("event_count")
)

In [6]:
gold = gold.withColumn(
    "criticality_score",
    (col("anomaly_rate") * 0.6) +
    (col("std_value") / col("avg_value") * 0.4)
)

In [ ]:
gold.write \
    .mode("overwrite") \
    .partitionBy("year", "month", "factory_id") \
    .parquet("s3a://gold/equipment_metrics_hourly/")

print("Camada Gold salva no MinIO: s3a://gold/equipment_metrics_hourly/")